In [1]:
# 📦 1. Instalar dependencias
!pip install -q transformers datasets peft accelerate

In [2]:
from google.colab import drive  # Importing the library to mount Google Drive
drive.mount('/content/drive')  # Mounting Google Drive in Colab environment

Mounted at /content/drive


In [3]:
import pandas as pd

# File paths
train_df_file = "/content/drive/My Drive/NEW_DGA_DETECTOR/train_1M.csv"

train_df = pd.read_csv(train_df_file)
train_df["Labels"]=train_df["label"]
print(train_df)

                                         domain   family  label Labels
0                               newmedialove.ru    legit  legit  legit
1                                   bankesb.com    legit  legit  legit
2                         sbjnvufhillsszger.net   fobber    dga    dga
3                                  rpltm.online    legit  legit  legit
4                               theblotsays.com    legit  legit  legit
...                                         ...      ...    ...    ...
1079995  c487bd0ebaf628e49016b23579812685.co.cc  bamital    dga    dga
1079996   c21377bf084803b5d03161903bc5f643.info  bamital    dga    dga
1079997                              yuncai.ltd    legit  legit  legit
1079998                          movingscam.com    legit  legit  legit
1079999                            aqaebif.info   pykspa    dga    dga

[1080000 rows x 4 columns]


In [4]:


# 📚 2. Importar librerías necesarias
import torch
import pandas as pd
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model
import os
os.environ["WANDB_DISABLED"] = "true"

# ⚙️ 3. Detectar dispositivo (GPU si está disponible)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔹 Usando dispositivo: {device.upper()}")

# 📑 4. Suponemos que ya tienes train_df cargado con columnas: domain y Label
# Por ejemplo, podrías cargarlo desde un CSV:
# train_df = pd.read_csv("tus_datos.csv")

# 🔤 5. Preprocesamiento
label_map = {"legit": 0, "dga": 1}
train_df["label"] = train_df["Labels"].map(label_map)
dataset = Dataset.from_pandas(train_df[["domain", "label"]])

# 🧠 6. Tokenizador y tokenización
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_fn(batch):
    return tokenizer(batch["domain"], truncation=True)

tokenized_dataset = dataset.map(tokenize_fn, batched=True)


🔹 Usando dispositivo: CUDA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1080000 [00:00<?, ? examples/s]

In [5]:
# 🏗️ 7. Cargar modelo base y aplicar LoRA
base_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],  # capas internas válidas para LoRA
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(base_model, lora_config)
model.to(device)

# 🛠️ 8. Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="no",
    per_device_train_batch_size=256,
    num_train_epochs=3,
    logging_steps=1000,
    save_strategy="no",
    learning_rate=2e-5,
    weight_decay=0.01
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 🏋️ 9. Entrenador (Corrección: eliminamos el argumento 'tokenizer' directo)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# 🚀 10. Entrenamiento
trainer.train()

# 💾 11. Guardar modelo
model.save_pretrained("./dga_detector_lora")
tokenizer.save_pretrained("./dga_detector_lora")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
1000,0.350538
2000,0.245396
3000,0.224075
4000,0.210914
5000,0.198723
6000,0.193056
7000,0.188255
8000,0.184293
9000,0.181583
10000,0.178791


('./dga_detector_lora/tokenizer_config.json',
 './dga_detector_lora/tokenizer.json')

In [6]:
from transformers import BertTokenizer, BertForSequenceClassification
from peft import PeftModel
import torch

# ✅ Cargar modelo y tokenizer
model_path = "./dga_detector_lora"  # Ruta donde guardaste el modelo
tokenizer = BertTokenizer.from_pretrained(model_path)

base_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model = PeftModel.from_pretrained(base_model, model_path)
model.eval()

# ✅ Detectar dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# ✅ Diccionario inverso de etiquetas
id2label = {0: "notdga", 1: "dga"}

# ✅ Función de predicción
def predict_domain(domain_name: str):
    inputs = tokenizer(domain_name, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_class = torch.argmax(outputs.logits, dim=1).item()
        return id2label[predicted_class]

# 🔍 Ejemplo
dominio = "google.com"
prediccion = predict_domain(dominio)
print(f"🔎 Dominio: {dominio} → Predicción: {prediccion}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🔎 Dominio: google.com → Predicción: notdga


In [7]:
import os
import time
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from transformers import BertTokenizer, BertForSequenceClassification
from peft import PeftModel

# ✅ Cargar modelo y tokenizer (ajustar ruta si es necesario)
model_path = "./dga_detector_lora"
tokenizer = BertTokenizer.from_pretrained(model_path)
base_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model = PeftModel.from_pretrained(base_model, model_path).eval().to("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Diccionario inverso
id2label = {0: "notdga", 1: "dga"}

# ✅ Función de predicción
def predict(domain):
    inputs = tokenizer(domain, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return predicted_class

# ✅ Familias que quieres evaluar
families = [
    'alureon.gz','bamital.gz', 'banjori.gz', 'bedep.gz', 'charbot.gz', 'chinad.gz',
    'conficker.gz', 'corebot.gz', 'cryptolocker.gz', 'deception.gz', 'dircrypt.gz',
    'dnschanger.gz', 'dyre.gz', 'emotet.gz', 'fobber.gz', 'gameover.gz', 'gozi.gz',
    'kraken.gz', 'locky.gz', 'manuelita.gz', 'matsnu.gz', 'monerominer.gz', 'murofet.gz',
    'murofetweekly.gz', 'mydoom.gz', 'necurs.gz', 'nymaim.gz', 'oderoor.gz', 'padcrypt.gz',
    'pitou.gz', 'proslikefan.gz', 'pushdo.gz', 'pykspa.gz', 'qadars.gz', 'qakbot.gz',
    'qsnatch.gz', 'ramdo.gz', 'ramnit.gz', 'ranbyus.gz', 'rovnix.gz', 'shiotob.gz',
    'simda.gz', 'sisron.gz', 'sphinx.gz', 'suppobox.gz', 'symmi.gz', 'tempedreve.gz',
    'tinba.gz', 'tinynuke.gz', 'vawtrak.gz', 'vidro.gz', 'virut.gz', 'zeus-newgoz.gz',
    'zloader.gz'
]

runs = 30
output_dir = "/content/drive/My Drive/results"
os.makedirs(output_dir, exist_ok=True)

# ✅ Evaluación por familia
for family in families:
    print(f"Processing family: {family}")

    # ⚠️ Asume que estos archivos tienen columnas: domain, label
    dga = pd.read_csv(f'/content/drive/My Drive/Familias_Test/{family}', chunksize=50)
    legit = pd.read_csv('/content/drive/My Drive/Familias_Test/legit.gz', chunksize=50)

    for run in range(runs):
        print(f'Run {run + 1}/{runs} - {family}', end='\r')

        # Obtener un chunk de cada uno
        df_dga = dga.get_chunk()
        df_legit = legit.get_chunk()

        # Agregar columna label
        df_dga["label"] = 1
        df_legit["label"] = 0

        # Mezclar y preparar
        dfw = pd.concat([df_dga, df_legit]).sample(frac=1).reset_index(drop=True)

        pred = []
        query_time = []

        for domain in dfw["domain"]:
            start = time.time()
            label = predict(domain)
            pred.append(label)
            query_time.append(time.time() - start)

        dfw["pred"] = pred
        dfw["query_time"] = query_time

        # ✅ Métricas (opcional)
        acc = accuracy_score(dfw["label"], dfw["pred"])
        f1 = f1_score(dfw["label"], dfw["pred"])
        print(f"Run {run+1} | ACC: {acc:.4f} | F1: {f1:.4f}")

        # ✅ Guardar resultados
        output_file = os.path.join(output_dir, f'results_LORA_classifier_{family}_{run}.csv.gz')
        dfw.to_csv(output_file, index=False)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Processing family: alureon.gz
Run 1 | ACC: 0.9900 | F1: 0.9901
Run 2 | ACC: 0.9900 | F1: 0.9901
Run 3 | ACC: 0.9300 | F1: 0.9307
Run 4 | ACC: 0.9600 | F1: 0.9608
Run 5 | ACC: 0.9700 | F1: 0.9703
Run 6 | ACC: 0.9900 | F1: 0.9901
Run 7 | ACC: 0.9900 | F1: 0.9901
Run 8 | ACC: 0.9900 | F1: 0.9901
Run 9 | ACC: 0.9900 | F1: 0.9899
Run 10 | ACC: 1.0000 | F1: 1.0000
Run 11 | ACC: 0.9900 | F1: 0.9899
Run 12 | ACC: 1.0000 | F1: 1.0000
Run 13 | ACC: 0.9700 | F1: 0.9703
Run 14 | ACC: 0.9600 | F1: 0.9600
Run 15 | ACC: 0.9600 | F1: 0.9600
Run 16 | ACC: 0.9800 | F1: 0.9804
Run 17 | ACC: 0.9800 | F1: 0.9804
Run 18 | ACC: 0.9800 | F1: 0.9804
Run 19 | ACC: 0.9300 | F1: 0.9320
Run 20 | ACC: 0.9900 | F1: 0.9901
Run 21 | ACC: 0.9800 | F1: 0.9804
Run 22 | ACC: 0.9200 | F1: 0.9216
Run 23 | ACC: 0.9500 | F1: 0.9495
Run 24 | ACC: 0.9800 | F1: 0.9796
Run 25 | ACC: 0.9700 | F1: 0.9709
Run 26 | ACC: 0.9800 | F1: 0.9796
Run 27 | ACC: 0.9600 | F1: 0.9592
Run 28 | ACC: 0.9800 | F1: 0.9800
Run 29 | ACC: 0.9800 | F1: 

In [8]:
import os
import time
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from transformers import BertTokenizer, BertForSequenceClassification
from peft import PeftModel

# ✅ Cargar modelo y tokenizer (ajustar ruta si es necesario)
model_path = "./dga_detector_lora"
tokenizer = BertTokenizer.from_pretrained(model_path)
base_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model = PeftModel.from_pretrained(base_model, model_path).eval().to("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Diccionario inverso
id2label = {0: "notdga", 1: "dga"}

# ✅ Función de predicción
def predict(domain):
    inputs = tokenizer(domain, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return predicted_class

# ✅ Familias que quieres evaluar
families = ['bazarbackdoor.gz',
            'bazarbackdoor_v2.gz',
            'bazarbackdoor_v3.gz',
            'bigviktor.gz',
            'bumblebee.gz',
            'ccleaner.gz',
            'dmsniff.gz',
            'enviserv.gz',
            'fosniw.gz',
            'goz.gz',
            'gozi_rfc4343.gz',
            'infy.gz',
            'monerodownloader.gz',
            'newgoz.gz',
            'ngioweb.gz',
            'pandabanker.gz',
            'pizd.gz',
            'reconyc.gz',
            'sharkbot.gz',
            'szribi.gz',
            'torpig.gz',
            'tufik.gz',
            'verblecon.gz',
            'wd.gz',
            'xshellghost.gz',
           ]

runs = 30
output_dir = "/content/drive/My Drive/results"
os.makedirs(output_dir, exist_ok=True)

# ✅ Evaluación por familia
for family in families:
    print(f"Processing family: {family}")

    # ⚠️ Asume que estos archivos tienen columnas: domain, label
    dga = pd.read_csv(f'/content/drive/My Drive/New_Families/{family}', chunksize=50)
    legit = pd.read_csv('/content/drive/My Drive/Familias_Test/legit.gz', chunksize=50)

    # Saltar los primeros 30 chunks de legit
    for _ in range(30):
        legit.get_chunk()


    for run in range(runs):
        print(f'Run {run + 1}/{runs} - {family}', end='\r')

        # Obtener un chunk de cada uno
        df_dga = dga.get_chunk()
        df_legit = legit.get_chunk()

        # Agregar columna label
        df_dga["label"] = 1
        df_legit["label"] = 0

        # Mezclar y preparar
        dfw = pd.concat([df_dga, df_legit]).sample(frac=1).reset_index(drop=True)

        pred = []
        query_time = []

        for domain in dfw["domain"]:
            start = time.time()
            label = predict(domain)
            pred.append(label)
            query_time.append(time.time() - start)

        dfw["pred"] = pred
        dfw["query_time"] = query_time

        # ✅ Métricas (opcional)
        acc = accuracy_score(dfw["label"], dfw["pred"])
        f1 = f1_score(dfw["label"], dfw["pred"])
        print(f"Run {run+1} | ACC: {acc:.4f} | F1: {f1:.4f}")

        # ✅ Guardar resultados
        output_file = os.path.join(output_dir, f'results_LORA_classifier_{family}_{run}.csv.gz')
        dfw.to_csv(output_file, index=False)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Processing family: bazarbackdoor.gz
Run 1 | ACC: 0.9600 | F1: 0.9583
Run 2 | ACC: 0.9500 | F1: 0.9495
Run 3 | ACC: 0.9600 | F1: 0.9592
Run 4 | ACC: 0.9900 | F1: 0.9899
Run 5 | ACC: 0.9500 | F1: 0.9505
Run 6 | ACC: 0.9600 | F1: 0.9600
Run 7 | ACC: 0.9400 | F1: 0.9400
Run 8 | ACC: 0.9800 | F1: 0.9796
Run 9 | ACC: 0.9900 | F1: 0.9899
Run 10 | ACC: 0.9500 | F1: 0.9505
Run 11 | ACC: 0.9500 | F1: 0.9485
Run 12 | ACC: 0.9400 | F1: 0.9400
Run 13 | ACC: 0.9700 | F1: 0.9691
Run 14 | ACC: 0.9600 | F1: 0.9592
Run 15 | ACC: 0.9600 | F1: 0.9608
Run 16 | ACC: 0.9700 | F1: 0.9703
Run 17 | ACC: 0.9300 | F1: 0.9278
Run 18 | ACC: 0.9400 | F1: 0.9388
Run 19 | ACC: 0.9900 | F1: 0.9899
Run 20 | ACC: 0.9700 | F1: 0.9703
Run 21 | ACC: 0.9600 | F1: 0.9600
Run 22 | ACC: 0.9400 | F1: 0.9400
Run 23 | ACC: 0.9500 | F1: 0.9495
Run 24 | ACC: 0.9600 | F1: 0.9608
Run 25 | ACC: 0.9500 | F1: 0.9524
Run 26 | ACC: 0.9700 | F1: 0.9703
Run 27 | ACC: 0.9400 | F1: 0.9400
Run 28 | ACC: 0.9400 | F1: 0.9388
Run 29 | ACC: 0.9500 

In [9]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Montar Google Drive (si aún no está montado)
from google.colab import drive
drive.mount('/content/drive')

# Función para calcular FPR y TPR
def fpr_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn + 1e-10)  # Evita división por cero
    tpr = tp / (tp + fn + 1e-10)
    return fpr, tpr

# Lista de familias de malware
families = [
    'alureon.gz','bamital.gz', 'banjori.gz', 'bedep.gz', 'charbot.gz', 'chinad.gz',
    'conficker.gz', 'corebot.gz', 'cryptolocker.gz', 'deception.gz', 'dircrypt.gz',
    'dnschanger.gz', 'dyre.gz', 'emotet.gz', 'fobber.gz', 'gameover.gz', 'gozi.gz',
    'kraken.gz', 'locky.gz', 'manuelita.gz', 'matsnu.gz', 'monerominer.gz', 'murofet.gz',
    'murofetweekly.gz', 'mydoom.gz', 'necurs.gz', 'nymaim.gz', 'oderoor.gz', 'padcrypt.gz',
    'pitou.gz', 'proslikefan.gz', 'pushdo.gz', 'pykspa.gz', 'qadars.gz', 'qakbot.gz',
    'qsnatch.gz', 'ramdo.gz', 'ramnit.gz', 'ranbyus.gz', 'rovnix.gz', 'shiotob.gz',
    'simda.gz', 'sisron.gz', 'sphinx.gz', 'suppobox.gz', 'symmi.gz', 'tempedreve.gz',
    'tinba.gz', 'tinynuke.gz', 'vawtrak.gz', 'vidro.gz', 'virut.gz', 'zeus-newgoz.gz',
    'zloader.gz'
]
runs = 30

# Listas para métricas globales
all_acc, all_pre, all_rec, all_f1 = [], [], [], []
all_fpr, all_tpr, all_qt, all_qts = [], [], [], []
total_unknowns_global = 0

for family in families:
    acc, pre, rec, f1, fpr, tpr, qt, qts = [], [], [], [], [], [], [], []
    total_unknowns = 0

    for run in range(runs):
        path = f'/content/drive/My Drive/results/results_LORA_classifier_{family}_{run}.csv.gz'
        if not os.path.exists(path):
            print(f"[⚠️] Archivo no encontrado: {path}")
            continue

        df = pd.read_csv(path)

        # Reemplazar -1 por 1 en 'pred', si aplica

        y_true = df['label']
        y_pred = df['pred']

        # Métricas
        acc.append(accuracy_score(y_true, y_pred))
        pre.append(precision_score(y_true, y_pred, zero_division=0))
        rec.append(recall_score(y_true, y_pred, zero_division=0))
        f1.append(f1_score(y_true, y_pred, zero_division=0))
        fpr_val, tpr_val = fpr_tpr(y_true, y_pred)
        fpr.append(fpr_val)
        tpr.append(tpr_val)

        if 'query_time' in df.columns:
            qt.append(df['query_time'].mean())
            qts.append(df['query_time'].std())

    # Promedios por familia
    if acc:  # solo si hubo archivos válidos
        print(f'{family.split(".")[0]:15}: '
              f'acc:{np.mean(acc):.2f}±{np.std(acc):.3f} '
              f'f1:{np.mean(f1):.2f}±{np.std(f1):.3f} '
              f'pre:{np.mean(pre):.2f}±{np.std(pre):.3f} '
              f'rec:{np.mean(rec):.2f}±{np.std(rec):.3f} '
              f'FPR:{np.mean(fpr):.2f}±{np.std(fpr):.3f} '
              f'TPR:{np.mean(tpr):.2f}±{np.std(tpr):.3f} '
              f'QT:{np.mean(qt):.5f}±{np.std(qt):.5f} '
              f'Unknowns: {total_unknowns}')

        all_acc.append(np.mean(acc))
        all_pre.append(np.mean(pre))
        all_rec.append(np.mean(rec))
        all_f1.append(np.mean(f1))
        all_fpr.append(np.mean(fpr))
        all_tpr.append(np.mean(tpr))
        all_qt.append(np.mean(qt))
        all_qts.append(np.mean(qts))
        total_unknowns_global += total_unknowns

# 🔍 Métricas globales
print("\n### 📊 Métricas globales ###")
print(f'Accuracy   : {np.mean(all_acc):.2f}')
print(f'F1-Score   : {np.mean(all_f1):.2f}')
print(f'Precision  : {np.mean(all_pre):.2f}')
print(f'Recall     : {np.mean(all_rec):.2f}')
print(f'FPR        : {np.mean(all_fpr):.2f}')
print(f'TPR        : {np.mean(all_tpr):.2f}')
print(f'Query time : {np.mean(all_qt):.5f} ± {np.mean(all_qts):.5f}')
print(f'Total unknown classifications: {total_unknowns_global}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
alureon        : acc:0.97±0.020 f1:0.97±0.020 pre:0.97±0.025 rec:0.98±0.021 FPR:0.03±0.027 TPR:0.98±0.021 QT:0.01243±0.00154 Unknowns: 0
bamital        : acc:0.98±0.013 f1:0.98±0.013 pre:0.97±0.024 rec:1.00±0.000 FPR:0.03±0.027 TPR:1.00±0.000 QT:0.01318±0.00162 Unknowns: 0
banjori        : acc:0.90±0.023 f1:0.89±0.026 pre:0.96±0.028 rec:0.83±0.043 FPR:0.03±0.027 TPR:0.83±0.043 QT:0.01273±0.00171 Unknowns: 0
bedep          : acc:0.98±0.014 f1:0.98±0.013 pre:0.97±0.024 rec:0.99±0.011 FPR:0.03±0.027 TPR:0.99±0.011 QT:0.01304±0.00166 Unknowns: 0
charbot        : acc:0.72±0.035 f1:0.62±0.060 pre:0.94±0.050 rec:0.47±0.064 FPR:0.03±0.027 TPR:0.47±0.064 QT:0.01283±0.00160 Unknowns: 0
chinad         : acc:0.98±0.013 f1:0.98±0.013 pre:0.97±0.024 rec:1.00±0.000 FPR:0.03±0.027 TPR:1.00±0.000 QT:0.01319±0.00174 Unknowns: 0
conficker      : acc:0.92±0.029 f1:0.91±0.031 pre

In [10]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Montar Google Drive (si aún no está montado)
from google.colab import drive
drive.mount('/content/drive')

# Función para calcular FPR y TPR
def fpr_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn + 1e-10)  # Evita división por cero
    tpr = tp / (tp + fn + 1e-10)
    return fpr, tpr

# Lista de familias de malware
families = ['bazarbackdoor.gz',
            'bazarbackdoor_v2.gz',
            'bazarbackdoor_v3.gz',
            'bigviktor.gz',
            'bumblebee.gz',
            'ccleaner.gz',
            'dmsniff.gz',
            'enviserv.gz',
            'fosniw.gz',
            'goz.gz',
            'gozi_rfc4343.gz',
            'infy.gz',
            'monerodownloader.gz',
            'newgoz.gz',
            'ngioweb.gz',
            'pandabanker.gz',
            'pizd.gz',
            'reconyc.gz',
            'sharkbot.gz',
            'szribi.gz',
            'torpig.gz',
            'tufik.gz',
            'verblecon.gz',
            'wd.gz',
            'xshellghost.gz',
           ]
runs = 30

# Listas para métricas globales
all_acc, all_pre, all_rec, all_f1 = [], [], [], []
all_fpr, all_tpr, all_qt, all_qts = [], [], [], []
total_unknowns_global = 0

for family in families:
    acc, pre, rec, f1, fpr, tpr, qt, qts = [], [], [], [], [], [], [], []
    total_unknowns = 0

    for run in range(runs):
        path = f'/content/drive/My Drive/results/results_LORA_classifier_{family}_{run}.csv.gz'
        if not os.path.exists(path):
            print(f"[⚠️] Archivo no encontrado: {path}")
            continue

        df = pd.read_csv(path)

        # Reemplazar -1 por 1 en 'pred', si aplica

        y_true = df['label']
        y_pred = df['pred']

        # Métricas
        acc.append(accuracy_score(y_true, y_pred))
        pre.append(precision_score(y_true, y_pred, zero_division=0))
        rec.append(recall_score(y_true, y_pred, zero_division=0))
        f1.append(f1_score(y_true, y_pred, zero_division=0))
        fpr_val, tpr_val = fpr_tpr(y_true, y_pred)
        fpr.append(fpr_val)
        tpr.append(tpr_val)

        if 'query_time' in df.columns:
            qt.append(df['query_time'].mean())
            qts.append(df['query_time'].std())

    # Promedios por familia
    if acc:  # solo si hubo archivos válidos
        print(f'{family.split(".")[0]:15}: '
              f'acc:{np.mean(acc):.2f}±{np.std(acc):.3f} '
              f'f1:{np.mean(f1):.2f}±{np.std(f1):.3f} '
              f'pre:{np.mean(pre):.2f}±{np.std(pre):.3f} '
              f'rec:{np.mean(rec):.2f}±{np.std(rec):.3f} '
              f'FPR:{np.mean(fpr):.2f}±{np.std(fpr):.3f} '
              f'TPR:{np.mean(tpr):.2f}±{np.std(tpr):.3f} '
              f'QT:{np.mean(qt):.5f}±{np.std(qt):.5f} '
              f'Unknowns: {total_unknowns}')

        all_acc.append(np.mean(acc))
        all_pre.append(np.mean(pre))
        all_rec.append(np.mean(rec))
        all_f1.append(np.mean(f1))
        all_fpr.append(np.mean(fpr))
        all_tpr.append(np.mean(tpr))
        all_qt.append(np.mean(qt))
        all_qts.append(np.mean(qts))
        total_unknowns_global += total_unknowns

# 🔍 Métricas globales
print("\n### 📊 Métricas globales ###")
print(f'Accuracy   : {np.mean(all_acc):.2f}')
print(f'F1-Score   : {np.mean(all_f1):.2f}')
print(f'Precision  : {np.mean(all_pre):.2f}')
print(f'Recall     : {np.mean(all_rec):.2f}')
print(f'FPR        : {np.mean(all_fpr):.2f}')
print(f'TPR        : {np.mean(all_tpr):.2f}')
print(f'Query time : {np.mean(all_qt):.5f} ± {np.mean(all_qts):.5f}')
print(f'Total unknown classifications: {total_unknowns_global}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bazarbackdoor  : acc:0.96±0.016 f1:0.96±0.016 pre:0.96±0.023 rec:0.95±0.024 FPR:0.04±0.024 TPR:0.95±0.024 QT:0.01271±0.00162 Unknowns: 0
bazarbackdoor_v2: acc:0.96±0.021 f1:0.96±0.021 pre:0.96±0.024 rec:0.96±0.029 FPR:0.04±0.024 TPR:0.96±0.029 QT:0.01273±0.00163 Unknowns: 0
bazarbackdoor_v3: acc:0.71±0.040 f1:0.61±0.072 pre:0.92±0.048 rec:0.46±0.079 FPR:0.04±0.024 TPR:0.46±0.079 QT:0.01993±0.00774 Unknowns: 0
bigviktor      : acc:0.60±0.035 f1:0.37±0.082 pre:0.86±0.094 rec:0.24±0.065 FPR:0.04±0.024 TPR:0.24±0.065 QT:0.01273±0.00182 Unknowns: 0
bumblebee      : acc:0.96±0.018 f1:0.96±0.018 pre:0.96±0.024 rec:0.95±0.029 FPR:0.04±0.024 TPR:0.95±0.029 QT:0.01293±0.00165 Unknowns: 0
ccleaner       : acc:0.93±0.028 f1:0.93±0.031 pre:0.96±0.025 rec:0.90±0.050 FPR:0.04±0.024 TPR:0.90±0.050 QT:0.01300±0.00163 Unknowns: 0
dmsniff        : acc:0.87±0.105 f1:0.84±0.146 p